# Validade Temporal e Supersessão de Fatos em Memória Associativa Hebbiana (HeLa-Mem)

**Projeto de Mestrado — Deep Learning (CIn / UFPE)**  
**Autor:** Filipe Augusto de Ataíde Barbosa (`faab@cin.ufpe.br`)  
**Artigo-Base:** *HeLa-Mem: Hebbian Learning and Associative Memory for LLM Agents* (ACL 2026, commit `68f19b3`)  
**Benchmark:** LongMemEval-S (500 instâncias, ICLR 2025)

--- 
### Objetivo deste Notebook
Executar a avaliação da **Proposta de Validade Temporal e Supersessão Explícita de Fatos** na Memória Semântica do HeLa-Mem, comparando diretamente com a **Baseline A oficial congelada em 61,20% (306/500)**.

> **Status Atual:** 200 itens já foram processados e avaliados com sucesso (Acurácia: **53,50% vs 50,50%** na baseline, com ganho de **+9,68 p.p.** em multi-session). O script retomará automaticamente a partir do item 200 até o item 500.

## 1. Configuração do Ambiente e Descompactação dos Dados Pré-computados

Se estiver executando no **Google Colab**:
1. Adicione sua chave da OpenAI no menu lateral **Secrets** (ícone de chave 🔑) com o nome **`OPENAI_API_KEY`** e habilite o acesso para este notebook.
2. Execute a célula abaixo para clonar o repositório, descompactar os 500 grafos episódicos da baseline e os 100 itens já processados da proposta.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Detectado ambiente Google Colab!")
    %cd /content
    if not os.path.exists("mestrado-dl-projeto"):
        !git clone --recurse-submodules https://github.com/ataidefilipe/mestrado-dl-projeto.git
    %cd /content/mestrado-dl-projeto
    !git pull
    !git submodule update --init --recursive
    PROJECT_ROOT = Path('/content/mestrado-dl-projeto')

    print("📦 Descompactando grafos da baseline (500 itens) e checkpoints da proposta (100 itens)...")
    !mkdir -p results/longmemeval_baseline_full results/longmemeval_temporal_full
    !unzip -q -o results/baseline_episodic_500.zip -d results/longmemeval_baseline_full/
    !unzip -q -o results/temporal_encoded_*.zip -d results/longmemeval_temporal_full/
    print("✅ Dados descompactados com sucesso!")

    print("📦 Instalando dependências...")
    !pip install -q sentence-transformers openai python-dotenv pandas tabulate

    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        print("🔑 OPENAI_API_KEY carregada com sucesso via Colab Secrets.")
    except Exception as e:
        print("⚠️ OPENAI_API_KEY não encontrada no Colab Secrets. Adicione nos Secrets (ícone de chave).")
else:
    PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
    print(f"Executando localmente no diretório: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'external' / 'HeLa-Mem'))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

assert os.environ.get('OPENAI_API_KEY'), "❌ ERRO: OPENAI_API_KEY não encontrada! Configure no .env ou no Colab Secrets."
print("Ambiente pronto e configurado!")

## 2. Verificação dos Módulos da Proposta B
Verifica se as extensões temporais (`TemporalKnowledgeMemory` e `ValidityAwareRetriever`) estão carregadas corretamente.

In [ ]:
from src.modeling.temporal_knowledge_memory import TemporalKnowledgeMemory
from src.modeling.validity_aware_retriever import ValidityAwareRetriever

print("✅ Módulos da Proposta B importados com sucesso!")

## 3. Execução da Pipeline (Itens 100 a 500)

O runner `run_temporal_hela_longmemeval.py` implementa **checkpointing automático**:
- Ele detecta os 100 itens já processados e pula imediatamente para o item 101.
- Reutiliza os 500 grafos episódicos da baseline (`--reuse_episodic_dir`), garantindo isolamento experimental e economia de tempo/custo de API.
- Loga o progresso a cada 10 itens com estimativa de tempo restante (ETA).
- Tempo estimado: **~50 a 60 minutos** com 5 workers simultâneos.

In [ ]:
# Configuração da execução
NUM_ITEMS = 500  # Total de 500 itens do LongMemEval-S
STAGE = 'all'    # Executa codificação semântica temporal e avaliação completa
WORKERS = 5

!python -u src/analysis/run_temporal_hela_longmemeval.py \
    --stage {STAGE} \
    --num_items {NUM_ITEMS} \
    --output_dir results/longmemeval_temporal_full \
    --reuse_episodic_dir results/longmemeval_baseline_full \
    --workers {WORKERS}

## 4. Comparação A/B Pareada e Relatório Final
Gera a tabela analítica pareada por categoria de questão comparando a Baseline A oficial (61,20%) com a Proposta B.

In [ ]:
import json
import pandas as pd

base_sum_path = PROJECT_ROOT / 'results' / 'longmemeval_baseline_full' / 'eval_results' / 'eval_summary.json'
prop_sum_path = PROJECT_ROOT / 'results' / 'longmemeval_temporal_full' / 'eval_results' / 'eval_summary.json'

with open(base_sum_path, 'r', encoding='utf-8') as f:
    base_data = json.load(f)

with open(prop_sum_path, 'r', encoding='utf-8') as f:
    prop_data = json.load(f)

print(f"=======================================================")
print(f"Acurácia Global - Baseline A (500 itens): {base_data.get('overall_accuracy', 0):.2f}%")
print(f"Acurácia Global - Proposta B (500 itens): {prop_data.get('overall_accuracy', 0):.2f}%")
diff = prop_data.get('overall_accuracy', 0) - base_data.get('overall_accuracy', 0)
print(f"Delta Global: {diff:+.2f} p.p.")
print(f"=======================================================")

# Comparação por Categoria
base_types = base_data.get('accuracy_by_type', {})
prop_types = prop_data.get('accuracy_by_type', {})

cat_rows = []
for qtype in sorted(set(list(base_types.keys()) + list(prop_types.keys()))):
    b_acc = base_types.get(qtype, {}).get('accuracy', 0.0)
    p_acc = prop_types.get(qtype, {}).get('accuracy', 0.0)
    p_tot = prop_types.get(qtype, {}).get('total', 0)
    cat_rows.append({
        'Categoria': qtype,
        'Total': p_tot,
        'Baseline A (%)': f"{b_acc:.2f}%",
        'Proposta B (%)': f"{p_acc:.2f}%",
        'Delta (p.p.)': f"{p_acc - b_acc:+.2f}"
    })

df_cat = pd.DataFrame(cat_rows)
display(df_cat)

# Análise Pareada Item a Item
base_by_id = {r['question_id']: r['correct'] for r in base_data.get('results', [])}
rows = []
for r in prop_data.get('results', []):
    qid = r['question_id']
    rows.append({
        'question_id': qid,
        'question_type': r['question_type'],
        'baseline_a': base_by_id.get(qid, None),
        'proposal_b': r['correct'],
        'generated_b': r.get('generated_answer', ''),
        'ground_truth': r.get('ground_truth', '')
    })

df = pd.DataFrame(rows)
df['delta'] = df['proposal_b'] - df['baseline_a']
print(f"\nDistribuição Pareada ({len(df)} itens avaliados):")
print(f"  📈 Ganhos (Baseline 0 -> Proposta 1): {len(df[df['delta'] > 0])}")
print(f"  📉 Regressões (Baseline 1 -> Proposta 0): {len(df[df['delta'] < 0])}")
print(f"  🔒 Mantidos Corretos (1 -> 1): {len(df[(df['proposal_b'] == 1) & (df['baseline_a'] == 1)])}")
print(f"  ❌ Mantidos Incorretos (0 -> 0): {len(df[(df['proposal_b'] == 0) & (df['baseline_a'] == 0)])}")

## 5. Download dos Resultados (Opcional - Google Colab)
Compacta a pasta de resultados finais para baixar para a máquina local ou persistir no Google Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive, files
    try:
        drive.mount('/content/drive')
        drive_dest = Path('/content/drive/MyDrive/mestrado_results_backup')
        drive_dest.mkdir(parents=True, exist_ok=True)
        !cp -ru results/longmemeval_temporal_full "{drive_dest}/"
        print(f"✅ Backup salvo no Google Drive: {drive_dest}")
    except Exception as e:
        print(f"Aviso Drive: {e}")
    output_zip = '/content/results_temporal_full_500.zip'
    !zip -q -r {output_zip} results/longmemeval_temporal_full/ reports/analysis/
    print(f"Arquivo ZIP gerado: {output_zip}")
    # files.download(output_zip)
